# 04 — Final analysis bound to the approved design
The notebook loads the immutable `DesignSpec` and exact calibration artifact from
storage. Analysis before the registered end is blocked; explicit interim mode returns
guardrails only. Estimator agreement is a sensitivity check, not independent evidence.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-2.0.0a2-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
import json
from datetime import timedelta
from pyspark.sql import functions as F

from supply_experiments.calibration.aa import AACalibration
from supply_experiments.design.spec import DesignSpec
from supply_experiments.io import (NATIONAL_HOLIDAYS, TABLES, load_city_panel,
                                   load_experiment, load_fixed_control)
from supply_experiments.panel import ExperimentWindow
from supply_experiments.reporting import analyze_experiment

dbutils.widgets.text("experiment_id", "")
EXPERIMENT_ID = dbutils.widgets.get("experiment_id")

tables = TABLES["sandbox"]
rec = load_experiment(spark, tables["registry"], EXPERIMENT_ID)
assert rec.status in {"completed", "analyzed"}, f"unexpected lifecycle state: {rec.status}"
spec = DesignSpec.from_json(rec.design_spec_json)
assert spec.fingerprint == rec.design_fingerprint

artifact_rows = (spark.table(tables["calibration"])
    .where(F.col("calibration_fingerprint") == rec.calibration_fingerprint)
    .where(F.col("passed") == F.lit(True))
    .orderBy(F.col("run_at").desc()).limit(1).collect())
assert artifact_rows, "matching passing calibration artifact not found"
aa = AACalibration.from_json(artifact_rows[0]["artifact_json"])
assert aa.matches_design(spec), 'calibration artifact does not match DesignSpec'

In [ ]:
date_start = str(
    rec.start_date - timedelta(days=rec.pre_window_days + spec.anticipation_days)
)
data_policy = json.loads(rec.config_json)["data_policy"]
panel, _ = load_city_panel(
    spark, date_start, str(rec.end_date), **data_policy,
    require_rupture_telemetry="rupture_rate_order" in rec.guardrail_kpis,
)
treated = list(spec.treated_units)
donors = list(spec.donor_units)
window = ExperimentWindow(
    start_date=spec.start_date, end_date=spec.end_date,
    pre_window_days=spec.pre_days, anticipation_days=spec.anticipation_days,
)

In [ ]:
# Optional complementary DiD: equivalence plus correlation on recent pre-data.
from supply_experiments.design.control_selection import revalidate_fixed_control

did_control = None
fixed = load_fixed_control(spark, tables["registry"])
if fixed:
    val = revalidate_fixed_control(
        panel, fixed, window_days=90, holidays=NATIONAL_HOLIDAYS,
        target_exclude=set(treated), end_date=window.pre_end,
        equivalence_margin=0.10, min_corr=0.80,
    )
    print(val.summary())
    if val.passed:
        did_control = list(fixed)
else:
    print("No current fixed control; complementary DiD omitted.")

In [ ]:
report = analyze_experiment(
    panel, rec.experiment_id, treated, donors, window,
    primary_kpi=spec.primary_kpi, guardrail_kpis=rec.guardrail_kpis,
    alpha=spec.alpha, max_group_placebos=spec.max_group_placebos,
    holidays=NATIONAL_HOLIDAYS, did_control=did_control,
    run_conformal=True, design_spec=spec, calibration=aa,
)
assert report.decision_eligible, report.warnings
print(report.summary())

In [ ]:
display(report.to_frame())
if not report.guardrails.empty:
    display(report.guardrails)

In [ ]:
import json
from datetime import datetime
rows = report.to_frame().assign(
    experiment_id=rec.experiment_id, kpi=rec.primary_kpi,
    decision=report.decision, decision_eligible=report.decision_eligible,
    design_fingerprint=report.design_fingerprint,
    calibration_fingerprint=report.calibration_fingerprint,
    sensitivity_agreement=report.triangulation, analyzed_at=datetime.utcnow(),
    warnings=json.dumps(report.warnings, ensure_ascii=False),
)
spark.createDataFrame(rows).write.format("delta").mode("append") \
    .option("mergeSchema", "true").saveAsTable(tables["results"])
print("Results persisted in", tables["results"])